In [4]:
from pathlib import Path
import pandas as pd

accepted_data_pay_gap_df = pd.read_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\accepted_pay_gap.csv"
)

accepted_data_pay_gap_df.shape

(49880, 17)

In [48]:
accepted_data_pay_gap_df['gender'] = accepted_data_pay_gap_df['gender'].str.strip()
accepted_data_pay_gap_df['gender'] = (accepted_data_pay_gap_df['gender'].replace({'Not Recorded':'Not recorded'}))
accepted_data_pay_gap_df['gender'].unique()

array(['Female', 'Male', 'Prefer not to say', 'Non-binary',
       'Not recorded'], dtype=object)

In [40]:
accepted_data_pay_gap_df['department'] = accepted_data_pay_gap_df['department'].str.strip()
accepted_data_pay_gap_df['department'].unique()

array(['Data', 'HR', 'Support', 'Operations', 'Product', 'Legal',
       'Marketing', 'Finance', 'Engineering', 'Sales'], dtype=object)

In [41]:
accepted_data_pay_gap_df['job_title'] = accepted_data_pay_gap_df['job_title'].str.strip()
accepted_data_pay_gap_df['job_title'].unique()

array(['Analyst', 'Manager', 'Lead', 'Specialist', 'Senior Analyst'],
      dtype=object)

In [43]:
accepted_data_pay_gap_df['employment_type'] = accepted_data_pay_gap_df['employment_type'].str.strip()
accepted_data_pay_gap_df['employment_type'].unique()

array(['Part-time', 'Fixed-term', 'Permanent'], dtype=object)

In [49]:
accepted_data_pay_gap_df['work_location'] = accepted_data_pay_gap_df['work_location'].str.strip()
accepted_data_pay_gap_df['work_location'] = accepted_data_pay_gap_df['work_location'].replace({'Not Recorded':'Not recorded'})
accepted_data_pay_gap_df['work_location'].unique()

array(['Birmingham', 'Bristol', 'London', 'Leeds', 'Manchester',
       'Glasgow', 'Remote', 'Not recorded'], dtype=object)

In [50]:
accepted_data_pay_gap_df['ethnicity'] = accepted_data_pay_gap_df['ethnicity'].str.strip()
#accepted_data_pay_gap_df['work_location'] = accepted_data_pay_gap_df['work_location'].replace({'Not Recorded':'Not recorded'})
accepted_data_pay_gap_df['ethnicity'].unique()

array(['Other', 'Black', 'White', 'Mixed', 'Prefer not to say', 'Asian'],
      dtype=object)

In [5]:
gender = accepted_data_pay_gap_df["gender"].str.strip().str.lower()

male_pay = accepted_data_pay_gap_df.loc[
    gender.eq("male"), "annual_base_pay_gbp"
].dropna()

female_pay = accepted_data_pay_gap_df.loc[
    gender.eq("female"), "annual_base_pay_gbp"
].dropna()

print(f"Male headcount: {len(male_pay):,}")
print(f"Female headcount: {len(female_pay):,}")

Male headcount: 24,074
Female headcount: 24,182


#### Minimum headcount for comparisons
##### Show the headcount for both groups. Suppress the comparison if either group has fewer than 10 employees contributing to the measure. A count of exactly 10 meets the threshold.
##### For bonus recipient rates, use each group's total headcount. For average bonus among recipients, use each group's number of bonus recipients.

In [51]:
minimum_headcount = 10

if min(len(male_pay), len(female_pay)) < minimum_headcount:
    print(f"Pay gaps suppressed. Each group needs at least {minimum_headcount} employees.")
else:
    male_mean = male_pay.mean()
    female_mean = female_pay.mean()
    male_median = male_pay.median()
    female_median = female_pay.median()

    print(f"Mean base pay: Male £{male_mean:,.2f}, Female £{female_mean:,.2f}")
    print(f"Median base pay: Male £{male_median:,.2f}, Female £{female_median:,.2f}")

    if male_mean != 0:
        mean_gap_pct = (male_mean - female_mean) / male_mean * 100
        print(f"Mean annual base-pay gap: {mean_gap_pct:.2f}%")
    else:
        print("Mean annual base-pay gap is undefined because the male mean is zero.")

    if male_median != 0:
        median_gap_pct = (male_median - female_median) / male_median * 100
        print(f"Median annual base-pay gap: {median_gap_pct:.2f}%")
    else:
        print("Median annual base-pay gap is undefined because the male median is zero.")

Mean base pay: Male £41,977.97, Female £42,062.40
Median base pay: Male £38,600.00, Female £38,700.00
Mean annual base-pay gap: -0.20%
Median annual base-pay gap: -0.26%


In [11]:
weekly_hours = accepted_data_pay_gap_df["contracted_hours_per_week"]
annual_hours = weekly_hours.where(weekly_hours.gt(0)) * 52

accepted_data_pay_gap_df["hourly_base_pay_gbp"] = (
    accepted_data_pay_gap_df["annual_base_pay_gbp"] / annual_hours
)

In [52]:
hourly_pay = accepted_data_pay_gap_df["hourly_base_pay_gbp"]

male_hourly_pay = hourly_pay.loc[gender.eq("male")].dropna()
female_hourly_pay = hourly_pay.loc[gender.eq("female")].dropna()

male_hourly_headcount = len(male_hourly_pay)
female_hourly_headcount = len(female_hourly_pay)

print(f"Male hourly-pay headcount: {male_hourly_headcount:,}")
print(f"Female hourly-pay headcount: {female_hourly_headcount:,}")

Male hourly-pay headcount: 24,074
Female hourly-pay headcount: 24,182


In [54]:
if min(male_hourly_headcount, female_hourly_headcount) < minimum_headcount:
    print(f"Hourly pay gaps suppressed. Each group needs {minimum_headcount} employees.")
else:
    male_hourly_mean = male_hourly_pay.mean()
    female_hourly_mean = female_hourly_pay.mean()
    male_hourly_median = male_hourly_pay.median()
    female_hourly_median = female_hourly_pay.median()

    print(f"Mean hourly pay: Male £{male_hourly_mean:,.2f}, Female £{female_hourly_mean:,.2f}")
    print(f"Median hourly pay: Male £{male_hourly_median:,.2f}, Female £{female_hourly_median:,.2f}")

    if male_hourly_mean != 0:
        hourly_mean_gap_pct = (male_hourly_mean - female_hourly_mean) / male_hourly_mean * 100
        print(f"Mean hourly pay gap: {hourly_mean_gap_pct:.2f}%")
    else:
        print("Mean hourly pay gap is undefined because the male mean is zero.")

    if male_hourly_median != 0:
        hourly_median_gap_pct = (
            (male_hourly_median - female_hourly_median) / male_hourly_median * 100
        )
        print(f"Median hourly pay gap: {hourly_median_gap_pct:.2f}%")
    else:
        print("Median hourly pay gap is undefined because the male median is zero.")

Mean hourly pay: Male £27.37, Female £27.37
Median hourly pay: Male £24.52, Female £24.51
Mean hourly pay gap: 0.00%
Median hourly pay gap: 0.03%


#### Full-time-equivalent (FTE) pay
##### This project uses 40 contracted hours per week as the full-time reference.
##### FTE annual base pay = (recorded annual base pay ÷ contracted weekly hours) × 40.
##### FTE pay gaps are reported separately as a sensitivity measure. The headline pay gaps use recorded annual base pay.

In [22]:
full_time_weekly_hours = 40

accepted_data_pay_gap_df["fte_base_pay_gbp"] = (
    accepted_data_pay_gap_df["hourly_base_pay_gbp"] * full_time_weekly_hours * 52
)

fte_pay = accepted_data_pay_gap_df["fte_base_pay_gbp"]
male_fte_pay = fte_pay.loc[gender.eq("male")].dropna()
female_fte_pay = fte_pay.loc[gender.eq("female")].dropna()

print(f"Male FTE headcount: {len(male_fte_pay):,}")
print(f"Female FTE headcount: {len(female_fte_pay):,}")

Male FTE headcount: 24,074
Female FTE headcount: 24,182


In [56]:
if min(len(male_fte_pay), len(female_fte_pay)) < minimum_headcount:
    print(f"FTE pay gaps suppressed. Each group needs at least {minimum_headcount} employees.")
else:
    male_fte_mean = male_fte_pay.mean()
    female_fte_mean = female_fte_pay.mean()
    male_fte_median = male_fte_pay.median()
    female_fte_median = female_fte_pay.median()

    print(f"Mean FTE pay: Male £{male_fte_mean:,.2f}, Female £{female_fte_mean:,.2f}")
    print(f"Median FTE pay: Male £{male_fte_median:,.2f}, Female £{female_fte_median:,.2f}")

    if male_fte_mean != 0:
        fte_mean_gap_pct = (male_fte_mean - female_fte_mean) / male_fte_mean * 100
        print(f"Mean FTE pay gap: {fte_mean_gap_pct:.2f}%")
    else:
        print("Mean FTE pay gap is undefined because the male mean is zero.")

    if male_fte_median != 0:
        fte_median_gap_pct = (male_fte_median - female_fte_median) / male_fte_median * 100
        print(f"Median FTE pay gap: {fte_median_gap_pct:.2f}%")
    else:
        print("Median FTE pay gap is undefined because the male median is zero.")

Mean FTE pay: Male £56,938.85, Female £56,937.53
Median FTE pay: Male £51,000.00, Female £50,986.67
Mean FTE pay gap: 0.00%
Median FTE pay gap: 0.03%


In [26]:
accepted_data_pay_gap_df["calculated_total_annual_pay_gbp"] = (
    accepted_data_pay_gap_df["annual_base_pay_gbp"]
    + accepted_data_pay_gap_df["annual_bonus_gbp"]
).round(2)

In [27]:
total_pay_difference = (
    accepted_data_pay_gap_df["calculated_total_annual_pay_gbp"]
    - accepted_data_pay_gap_df["total_annual_pay_gbp"]
).abs().round(2)

#### Total annual pay validation
##### Calculated total annual pay equals annual base pay plus annual bonus.
##### Compare the calculated amount with the recorded total annual pay. An absolute difference of £0.01 or less passes. A difference greater than £0.01 fails.

In [28]:
# validation 
within_tolerance = total_pay_difference.le(0.01).fillna(False)

print(f"Rows within the £0.01 tolerance: {within_tolerance.sum():,}")
print(f"Rows that failed the check: {(~within_tolerance).sum():,}")

Rows within the £0.01 tolerance: 49,880
Rows that failed the check: 0


In [29]:
bonus_df = accepted_data_pay_gap_df[["annual_bonus_gbp"]].copy()
bonus_df["gender_group"] = gender
bonus_df = bonus_df.loc[bonus_df["gender_group"].isin(["male", "female"])].copy()

bonus_df["received_bonus"] = bonus_df["annual_bonus_gbp"].gt(0)
bonus_df["recipient_bonus_gbp"] = bonus_df["annual_bonus_gbp"].where(
    bonus_df["received_bonus"]
)

In [30]:
bonus_summary = (
    bonus_df.groupby("gender_group")
    .agg(
        headcount=("received_bonus", "size"),
        bonus_recipients=("received_bonus", "sum"),
        average_bonus_among_recipients_gbp=("recipient_bonus_gbp", "mean"),
    )
    .reindex(["male", "female"])
)

In [31]:
count_columns = ["headcount", "bonus_recipients"]
bonus_summary[count_columns] = bonus_summary[count_columns].fillna(0).astype(int)

bonus_summary["bonus_recipient_rate_pct"] = (
    bonus_summary["bonus_recipients"] / bonus_summary["headcount"] * 100
)

In [32]:
if bonus_summary["headcount"].min() < minimum_headcount:
    bonus_summary["bonus_recipient_rate_pct"] = pd.NA

if bonus_summary["bonus_recipients"].min() < minimum_headcount:
    bonus_summary["average_bonus_among_recipients_gbp"] = pd.NA

display(bonus_summary.round(2))

,headcount,bonus_recipients,average_bonus_among_recipients_gbp,bonus_recipient_rate_pct
gender_group,,,,
male,24074,24074,3559.46,100.0
female,24182,24182,3570.58,100.0


#### Questions it solved :
#### What percentage received a bonus?
#### Among those who received one, how much did they get on average?

In [33]:
import os

print(os.getcwd())

C:\Users\madha\anaconda_projects\bc8c0071-3af2-482e-87be-2d8530e1afc0


In [60]:
accepted_data_pay_gap_df.to_csv(
    r"C:\Users\madha\Desktop\Projects\UK_Gender_PayGap_BonusOutcomes_and_Representation Analytics\processed\standardised_pay_gap.csv",
index=False)
print(f"Accepted rows saved: {len(accepted_data_pay_gap_df):,}")

Accepted rows saved: 49,880
